In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl


In [2]:
import os
import csv
import json
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
from tqdm.auto import tqdm

# Быстрый json-парсер (если доступен)
try:
    import orjson  # type: ignore
except Exception:
    orjson = None


In [3]:
# ----------------------------
# Объём данных (trade-off quality vs speed)
# ----------------------------
# None -> использовать весь train (дольше)
MAX_SESSIONS_POP = 5_000_000
MAX_SESSIONS_COVISIT = 5_000_000

# ----------------------------
# Параметры сессии / пар
# ----------------------------
MAX_EVENTS_IN_SESSION = 30            # берем только хвост сессии
NEXT_PAIR_WINDOW = 20                # пары i -> i+1..i+NEXT_PAIR_WINDOW (ускоряет)
TIME_WINDOW_MS = 24 * 60 * 60 * 1000 # 1 день для click2click

# ----------------------------
# Размер item-item соседей
# ----------------------------
MAX_NEIGHBORS_CLICK = 120
MAX_NEIGHBORS_BUY = 200
PRUNE_AT_CLICK = MAX_NEIGHBORS_CLICK * 4
PRUNE_AT_BUY = MAX_NEIGHBORS_BUY * 4

# ----------------------------
# Типы событий (как в OTTO)
# ----------------------------
TYPE_MAP = {"clicks": 0, "carts": 1, "orders": 2}
INV_TYPE = {0: "clicks", 1: "carts", 2: "orders"}

# ----------------------------
# Веса для микса соседей по типу задачи
# ----------------------------
# target: clicks / carts / orders
# внутри: (w_click2click, w_buy2buy)
MIX_WEIGHTS = {
    0: (1.0, 0.2),  # clicks
    1: (0.6, 1.0),  # carts
    2: (0.4, 1.2),  # orders
}

# Усиление "последних" айтемов в сессии (recency)
RECENCY_SELF_BOOST = 2.0


In [4]:
def find_otto_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/otto-recommender-system"),
        Path("/kaggle/input/otto-recommender-system"),
        Path("/kaggle/input/otto-recsys-dataset"),
        Path("/kaggle/input/recsys-dataset"),
    ]
    for c in candidates:
        if (c / "train.jsonl").exists() and (c / "test.jsonl").exists():
            return c

    # fallback: рекурсивный поиск
    root = Path("/kaggle/input")
    if root.exists():
        for tr in root.rglob("train.jsonl"):
            d = tr.parent
            if (d / "test.jsonl").exists():
                return d

    raise FileNotFoundError("Не найдено train.jsonl/test.jsonl в /kaggle/input. Проверьте Input в Kaggle.")


DATA_DIR = find_otto_data_dir()
TRAIN_PATH = DATA_DIR / "train.jsonl"
TEST_PATH = DATA_DIR / "test.jsonl"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR:", DATA_DIR)
print("TRAIN:", TRAIN_PATH.exists(), TRAIN_PATH)
print("TEST :", TEST_PATH.exists(), TEST_PATH)
print("SAMPLE:", SAMPLE_SUB_PATH.exists(), SAMPLE_SUB_PATH)


DATA_DIR: /kaggle/input/competitions/otto-recommender-system
TRAIN: True /kaggle/input/competitions/otto-recommender-system/train.jsonl
TEST : True /kaggle/input/competitions/otto-recommender-system/test.jsonl
SAMPLE: True /kaggle/input/competitions/otto-recommender-system/sample_submission.csv


In [5]:
def _loads(line: bytes):
    if orjson is not None:
        return orjson.loads(line)
    return json.loads(line)


def prune_row_inplace(row: Dict[int, float], keep_n: int) -> None:
    # Оставляет top-N соседей по весу (in-place).
    if len(row) <= keep_n:
        return
    top = sorted(row.items(), key=lambda x: x[1], reverse=True)[:keep_n]
    row.clear()
    row.update(top)


In [ ]:
def build_pop_and_covisitation(
    train_path: Path,
    *,
    max_sessions_pop: Optional[int],
    max_sessions_covis: Optional[int],
) -> Tuple[Dict[int, List[int]], Dict[int, Dict[int, float]], Dict[int, Dict[int, float]]]:
    # popularity отдельно по типам
    pop_counts = {
        0: defaultdict(int),
        1: defaultdict(int),
        2: defaultdict(int),
    }

    covis_click: Dict[int, Dict[int, float]] = defaultdict(lambda: defaultdict(float))
    covis_buy: Dict[int, Dict[int, float]] = defaultdict(lambda: defaultdict(float))

    n_pop = 0
    n_cov = 0

    with open(train_path, "rb") as f:
        for line in tqdm(f, desc="Build pop + covis from train"):
            obj = _loads(line)
            events_raw = obj["events"]

            # компактный формат: [(aid, ts, type_id), ...]
            events: List[Tuple[int, int, int]] = [
                (int(e["aid"]), int(e["ts"]), TYPE_MAP[e["type"]]) for e in events_raw
            ]

            if not events:
                continue

            # ограничиваем хвост сессии
            events = events[-MAX_EVENTS_IN_SESSION:]

            # ----------------------------
            # Popularity (по типам)
            # ----------------------------
            if (max_sessions_pop is None) or (n_pop < max_sessions_pop):
                for aid, ts, t in events:
                    pop_counts[t][aid] += 1
                n_pop += 1

            # ----------------------------
            # Co-visitation (две матрицы)
            # ----------------------------
            if (max_sessions_covis is None) or (n_cov < max_sessions_covis):
                # click2click: учитываем только клики
                click_events = [(aid, ts) for aid, ts, t in events if t == 0]

                # Для скорости считаем пары только в пределах NEXT_PAIR_WINDOW
                for i, (ai, ti) in enumerate(click_events):
                    # пары с ближайшими следующими событиями
                    for aj, tj in click_events[i + 1 : i + 1 + NEXT_PAIR_WINDOW]:
                        # окно по времени (если события отсортированы по ts, можно делать break)
                        if (tj - ti) > TIME_WINDOW_MS:
                            break
                        covis_click[ai][aj] += 1.0
                        covis_click[aj][ai] += 1.0

                    if len(covis_click[ai]) > PRUNE_AT_CLICK:
                        prune_row_inplace(covis_click[ai], MAX_NEIGHBORS_CLICK)

                # buy2buy: carts + orders, берем уникальные aid (обычно их мало)
                seen = set()
                buy_aids: List[int] = []
                for aid, ts, t in events:
                    if t in (1, 2) and aid not in seen:
                        buy_aids.append(aid)
                        seen.add(aid)

                # пары всех buy-айтемов (маленький список)
                for i, ai in enumerate(buy_aids):
                    for aj in buy_aids[i + 1 :]:
                        covis_buy[ai][aj] += 1.0
                        covis_buy[aj][ai] += 1.0

                    if len(covis_buy[ai]) > PRUNE_AT_BUY:
                        prune_row_inplace(covis_buy[ai], MAX_NEIGHBORS_BUY)

                n_cov += 1

            # (опционально) ранний выход, если всё добрали
            if (max_sessions_pop is not None) and (max_sessions_covis is not None):
                if (n_pop >= max_sessions_pop) and (n_cov >= max_sessions_covis):
                    break

    # Финальная обрезка соседей
    for a in list(covis_click.keys()):
        prune_row_inplace(covis_click[a], MAX_NEIGHBORS_CLICK)
    for a in list(covis_buy.keys()):
        prune_row_inplace(covis_buy[a], MAX_NEIGHBORS_BUY)

    # Top popular (по типам)
    top_pop_by_type: Dict[int, List[int]] = {}
    for t in (0, 1, 2):
        top_pop_by_type[t] = [
            a for a, _ in sorted(pop_counts[t].items(), key=lambda x: x[1], reverse=True)[:500]
        ]

    return top_pop_by_type, covis_click, covis_buy


top_pop_by_type, covis_click, covis_buy = build_pop_and_covisitation(
    TRAIN_PATH,
    max_sessions_pop=MAX_SESSIONS_POP,
    max_sessions_covis=MAX_SESSIONS_COVISIT,
)

print("Top popular sizes:", {k: len(v) for k, v in top_pop_by_type.items()})
print("covis_click keys:", len(covis_click))
print("covis_buy   keys:", len(covis_buy))


Build pop + covis from train: 0it [00:00, ?it/s]

In [ ]:
def recommend_for_prefix(
    prefix_events: List[Tuple[int, int, int]],
    *,
    target_type_id: int,
    covis_click: Dict[int, Dict[int, float]],
    covis_buy: Dict[int, Dict[int, float]],
    top_pop_by_type: Dict[int, List[int]],
    k: int = 20,
) -> List[int]:
    # последние aid в порядке "с конца", но уникальные
    aids = [a for a, ts, t in prefix_events]
    seen = set()
    recent: List[int] = []
    for a in reversed(aids):
        if a not in seen:
            recent.append(a)
            seen.add(a)
        if len(recent) == 20:
            break

    # Начнем список рекомендаций с последних айтемов (часто помогает)
    recs: List[int] = []
    used = set()

    for a in recent:
        if a not in used:
            recs.append(a)
            used.add(a)
        if len(recs) == k:
            return recs

    # Счётчик баллов для кандидатов
    scores = defaultdict(float)

    w_click, w_buy = MIX_WEIGHTS[target_type_id]

    for r, a in enumerate(recent):
        # boost для самого айтема (recency)
        scores[a] += RECENCY_SELF_BOOST / (r + 1)

        # соседи click2click
        for nb, w in covis_click.get(a, {}).items():
            scores[nb] += w_click * w

        # соседи buy2buy
        for nb, w in covis_buy.get(a, {}).items():
            scores[nb] += w_buy * w

    # Ранжируем кандидатов, исключая уже добавленные
    ranked = [a for a, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True) if a not in used]

    for a in ranked:
        recs.append(a)
        used.add(a)
        if len(recs) == k:
            return recs

    # Добиваем популярными по типу
    for a in top_pop_by_type[target_type_id]:
        if a not in used:
            recs.append(a)
            used.add(a)
            if len(recs) == k:
                return recs

    # Фоллбек (на всякий)
    if len(recs) < k:
        for a in top_pop_by_type[target_type_id]:
            recs.append(a)
            if len(recs) == k:
                break

    return recs[:k]


In [ ]:
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

def write_submission(
    test_path: Path,
    out_path: Path,
    *,
    covis_click: Dict[int, Dict[int, float]],
    covis_buy: Dict[int, Dict[int, float]],
    top_pop_by_type: Dict[int, List[int]],
):
    with open(out_path, "w", newline="") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(["session_type", "labels"])

        with open(test_path, "rb") as f_in:
            for line in tqdm(f_in, desc="Write submission"):
                obj = _loads(line)
                session = int(obj["session"])
                events_raw = obj["events"]

                prefix_events: List[Tuple[int, int, int]] = [
                    (int(e["aid"]), int(e["ts"]), TYPE_MAP[e["type"]]) for e in events_raw
                ]
                prefix_events = prefix_events[-MAX_EVENTS_IN_SESSION:]

                for target_type_id in (0, 1, 2):
                    recs = recommend_for_prefix(
                        prefix_events,
                        target_type_id=target_type_id,
                        covis_click=covis_click,
                        covis_buy=covis_buy,
                        top_pop_by_type=top_pop_by_type,
                        k=20,
                    )
                    labels = " ".join(str(a) for a in recs)
                    writer.writerow([f"{session}_{INV_TYPE[target_type_id]}", labels])

write_submission(
    TEST_PATH,
    SUBMISSION_PATH,
    covis_click=covis_click,
    covis_buy=covis_buy,
    top_pop_by_type=top_pop_by_type,
)

print("Saved:", SUBMISSION_PATH, "exists:", SUBMISSION_PATH.exists())


In [ ]:
import pandas as pd

sub = pd.read_csv(SUBMISSION_PATH)
print("submission rows:", len(sub))

sample = pd.read_csv(SAMPLE_SUB_PATH) if SAMPLE_SUB_PATH.exists() else None
if sample is not None:
    print("sample rows    :", len(sample))

display(sub.head(3))

# 20 лейблов в каждой строке
lens = sub["labels"].astype(str).str.split().str.len()
print("min labels:", lens.min(), "max labels:", lens.max())
assert (lens == 20).all()

# нет пустых строк / NaN
assert sub["labels"].isna().sum() == 0
assert (sub["labels"].str.len() > 0).all()

# строгий паттерн: 20 целых чисел через пробел
assert sub["labels"].str.fullmatch(r"(?:\d+\s){19}\d+").notna().all()

# уникальность ключа
assert sub["session_type"].is_unique

if sample is not None:
    assert len(sub) == len(sample)

print("OK: submission format looks valid.")
